# Curating systems for SLiM simulations

**Useful commands**

Submitting many slurm scripts
* In system dir: ``for d in */; do (cd "$d" && sbatch gpu.sh); done``
* In sampling dir: ``for d in */; do (cd "$d" && (for s in */; do (cd "$s" && sbatch gpu.sh); done)); done``

Submitting slurm scripts where no ``kd.csv`` files exist
* In system dir: ``for d in */; do (cd "$d" && [ ! -f kd.csv ] && sbatch gpu.sh); done``
* In sampling dir: ``for d in */; do (cd "$d" && (for s in */; do (cd "$s" && [ ! -f kd.csv ] && sbatch gpu.sh); done)); done``

Checking number of frames
* From simulation dir: ``head -c 12 *.dcd | tail -c 4 | od -An -t dI``

In [1]:
import json
import os
import pandas as pd
import numpy as np
import subprocess
import shutil

while os.path.split(os.getcwd())[-1] != 'calvados-slim':
    os.chdir('..')

from src import curation, utils

/groups/sbinlab/fknudsen/software/miniconda3/envs/calvados/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Curating simulations

In [2]:
# Loading curations
with open('data/curation/systems.json') as file:
    systems = json.load(file)
systems_dir = os.path.abspath('data/curation/systems')

# Loading configurations
with open('data/configurations/conditions.json') as file:
    conditions = json.load(file)
with open('data/configurations/sampling.json') as file:
    sampling = json.load(file)
with open('data/configurations/go_scans.json') as file:
    go_scans = json.load(file)
slurm_scripts_dir = 'data/configurations/slurm'

In [3]:
# Setting modes to create runs for
conditions_mode = 'Default'
sampling_mode = 'Speed'

sim_dir = os.path.abspath(f"sim")

In [4]:
# Defining eps_go range
eps_go_min  = 0
eps_go_max  = 5
eps_go_step = 1
eps_gos = np.arange(eps_go_min, eps_go_max+eps_go_step, eps_go_step)
eps_gos = [5]

In [ ]:
# Generating simulation setups
replicates = 1

# Using date as hash
hash = pd.Timestamp.now('utc').strftime("%Y%m%d")

platform = input('Which platform [CPU/GPU]?').upper()

# Looping over systems
for system in systems:

    system_dir = os.path.join(systems_dir, system['ID'])

    if not os.path.exists(system_dir):
        continue

    # Looping over Go k scan
    for eps_go in eps_gos:

        # Creating eps_go directory
        eps_dir = os.path.join(sim_dir, system['ID'], f"eps_go={eps_go:.2f}")

        # Creating replica directory
        for rep in range(1, replicates+1):

            rep_dir = os.path.join(eps_dir, sampling_mode.lower(), f"rep{rep}")
            if os.path.exists(rep_dir):
                continue

            print(f"Creating '{rep_dir}'")
            try:
                curation.Simulation.build(
                    system,
                    system_dir,
                    conditions[conditions_mode],
                    sampling[sampling_mode],
                    eps_go, rep_dir, platform
                )
            except Exception as e:
                print(f"Failed to create {rep_dir}: {e}")
                shutil.rmtree(rep_dir)



## Inspecting simulations

In [ ]:
# Parsing simulations
sims = []
sim_path = os.path.abspath('sim')
for system in os.listdir(sim_path):
    system_path = os.path.join(sim_path, system)
    for eps_dir in os.listdir(system_path):
        eps_go = float(eps_dir.split('=')[-1])
        eps_path = os.path.join(system_path, eps_dir)
        for sampling_dir in os.listdir(eps_path):
            sampling_path = os.path.join(eps_path, sampling_dir)
            for rep_dir in os.listdir(sampling_path):
                rep = int(rep_dir[3:])
                rep_path = os.path.join(sampling_path, rep_dir)
                sims.append({
                    'System': system,
                    'eps_go': eps_go,
                    'Sampling mode': sampling_dir.capitalize(),
                    'Replicate': rep,
                    'Frames': utils.get_dcd_frame_count(os.path.join(rep_path, f'{system}.dcd')),
                    'Speed': utils.get_log_speed(os.path.join(rep_path, f'{system}.log')),
                    'Path': rep_path,
                })
sims = pd.DataFrame(sims).sort_values(['System','eps_go', 'Replicate']).reset_index(drop=True)
sims

### Evaluating need for replicas

In [ ]:
# Estimating need for replicates
speed_system = sims.groupby('System')['Speed'].mean()
goal_sim_time = 5000 # ns
goal_wall_time = 5 # Days
total_wall_time = (goal_sim_time / speed_system)
necessary_replicates = (total_wall_time / goal_wall_time).map(np.ceil).rename("Replicates")
necessary_replicates.to_csv('data/curation/system_replicates.csv')
necessary_replicates.sort_values(ascending=False)

## Submitting simulations

In [14]:
# Selecting simulations for submission

# Criteria options
not_simulated = lambda row: row['Frames'] == None
running = lambda row: row['Frames'] < 10000
whole_eps_go = lambda row: row['eps_go'] % 1 == 0
specific_eps_gos = lambda ks: lambda row: row['eps_go'] in [round(f, 1) for f in ks]
specific_system = lambda sys: lambda row: row['System'] == sys

criteria = [
    not_simulated
] 

sims_sel = sims.copy()
for criterion in criteria:
    sims_sel = sims_sel[sims_sel.apply(criterion, axis=1)]
sims_sel

,System,eps_go,Sampling mode,Replicate,Frames,Speed,Path
0,afadin-bcr,5.0,Speed,1,None,None,/lustre/hpc/sbinlab/fknudsen/calvados-slim/sim...
1,ankra2-ccdc8,5.0,Speed,1,None,None,/lustre/hpc/sbinlab/fknudsen/calvados-slim/sim...
2,ankra2-rfx7,5.0,Speed,1,None,None,/lustre/hpc/sbinlab/fknudsen/calvados-slim/sim...
3,ap1m1-tgoln2,5.0,Speed,1,None,None,/lustre/hpc/sbinlab/fknudsen/calvados-slim/sim...
4,ap2a-amph,5.0,Speed,1,None,None,/lustre/hpc/sbinlab/fknudsen/calvados-slim/sim...
...,...,...,...,...,...,...,...
273,wdr5-setd1a,5.0,Speed,1,None,None,/lustre/hpc/sbinlab/fknudsen/calvados-slim/sim...
274,wdr5-setd1b,5.0,Speed,1,None,None,/lustre/hpc/sbinlab/fknudsen/calvados-slim/sim...
275,wds-dgt1,5.0,Speed,1,None,None,/lustre/hpc/sbinlab/fknudsen/calvados-slim/sim...
276,wds-nsl1,5.0,Speed,1,None,None,/lustre/hpc/sbinlab/fknudsen/calvados-slim/sim...


In [ ]:
# Submitting simulations
for _, sim in sims_sel.iterrows():
    break # COMMENT AWAY MANUALLY - SUBMIT WITH CARE
    subprocess.run(
        ['sbatch', 'deic_cpu.sh'],
        cwd=sim['Path']
    )

## Assessing simulation speed